# TruthLens: Hybrid Fake News Detection Notebook

This notebook trains and evaluates a hybrid BERT + BiLSTM model for fake news detection and compares it against traditional machine learning baselines.

You will:
- prepare and split the WELFake dataset
- train the hybrid deep model with iteration-aware tracking
- evaluate performance with detailed diagnostics
- compare traditional ML models with the hybrid model
- save trained artifacts and metrics for deployment

## 0. Setup

This section initializes the environment, imports dependencies, sets reproducible seeds, and selects the best available runtime device.

## 0.1 Kaggle P100 CUDA Fix (Run Once)

Use this section only when Kaggle assigns a P100 and you see `no kernel image is available`.

Steps:
1. Set `REINSTALL_TORCH_P100 = True` in the next cell.
2. Run the next cell once.
3. Restart the Kaggle kernel/session.
4. Set `REINSTALL_TORCH_P100 = False` and run all cells from the top.

In [ ]:
import sys
import subprocess

# On T4/L4/A100 keep this False. Use True only for P100 + no-kernel-image CUDA error.
REINSTALL_TORCH_P100 = False

def _run(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

try:
    import torch
    _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print("Detected device:", _gpu_name)
except Exception:
    _gpu_name = "Unknown"

if REINSTALL_TORCH_P100:
    print("Installing torch/cu118 versions compatible with Kaggle P100...")
    _run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])
    _run([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "--index-url", "https://download.pytorch.org/whl/cu118",
        "torch==2.3.1", "torchvision==0.18.1", "torchaudio==2.3.1"
    ])
    print("Install complete. Restart kernel/session now, then rerun from Cell 1.")
else:
    print("Skipped reinstall. Set REINSTALL_TORCH_P100 = True only for P100 compatibility fix.")

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        print("capability:", torch.cuda.get_device_capability(0))
except Exception as e:
    print("Torch check warning:", type(e).__name__, str(e))

In [ ]:
import os
from pathlib import Path
import glob

import torch, torch.nn as nn, torch.nn.functional as F
torch.backends.cudnn.enabled = False  # fix LSTM CUDA kernel error
import pandas as pd, numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import BertModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve
    )
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
import warnings, random, time
warnings.filterwarnings('ignore')

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def is_kaggle_runtime():
    return 'KAGGLE_URL_BASE' in os.environ or Path('/kaggle').exists()

def resolve_data_path():
    # Priority: explicit env var > known Kaggle paths > Kaggle recursive search > local project defaults
    env_path = os.environ.get('WELFAKE_CSV_PATH')
    if env_path and Path(env_path).exists():
        return env_path

    candidates = [
        '/kaggle/input/datasets/kfsurya/welfake/preprocessed_welfake.csv',
        '/kaggle/input/welfake/preprocessed_welfake.csv',
        '/kaggle/input/**/preprocessed_welfake.csv',
        '/kaggle/input/**/WELFake_Dataset.csv',
        str(Path.cwd() / 'Data' / 'WELFake_Dataset.csv'),
        str(Path.cwd() / 'Data' / 'preprocessed_welfake.csv'),
    ]

    # 1) Direct existing paths first
    for p in candidates:
        if '*' not in p and Path(p).exists():
            return p

    # 2) Expand wildcard Kaggle patterns
    for p in candidates:
        if '*' in p:
            matches = sorted(glob.glob(p, recursive=True))
            if matches:
                return matches[0]

    # 3) Final fallback: scan Kaggle input for likely filenames
    if Path('/kaggle/input').exists():
        fallback = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))
        preferred = [x for x in fallback if 'welfake' in x.lower() or 'preprocessed' in x.lower()]
        if preferred:
            return preferred[0]

    raise FileNotFoundError(
        'Could not find dataset CSV. Set WELFAKE_CSV_PATH to your CSV file path, '
        'or verify your Kaggle dataset is attached under /kaggle/input.'
    )

def resolve_save_dir():
    # Priority: explicit env var > Kaggle working dir > local outputs dir
    env_dir = os.environ.get('TRUTHLENS_SAVE_DIR')
    if env_dir:
        return env_dir
    if is_kaggle_runtime():
        return '/kaggle/working/truthlens_model'
    return str(Path.cwd() / 'outputs' / 'truthlens_model')

def build_runtime_note(device, note):
    lines = [f'device: {device}', f'torch: {torch.__version__}', note]
    lower = note.lower()
    if device.type == 'cpu' and ('no kernel image' in lower or 'cuda fallback' in lower):
        lines.append('GPU advice: current CUDA/PyTorch build is incompatible with this GPU.')
        lines.append('Recommended on Kaggle: switch accelerator to T4, L4, or A100.')
        lines.append('If you must use P100, install a torch build that supports sm_60.')
    return ' | '.join(lines)

def pick_device():
    if os.environ.get('FORCE_CPU', '0') == '1':
        return torch.device('cpu'), 'FORCE_CPU=1'
    if not torch.cuda.is_available():
        return torch.device('cpu'), 'CUDA not available'
    try:
        # Small CUDA op to verify this GPU can run the installed torch CUDA kernels.
        x = torch.tensor([1.0], device='cuda')
        _ = x * 2.0
        torch.cuda.synchronize()
        name = torch.cuda.get_device_name(0)
        cc = torch.cuda.get_device_capability(0)
        gpu_note = f'{name} (cc {cc[0]}.{cc[1]})'
        if cc[0] < 7:
            gpu_note += ' | Note: older GPU arch; newer CUDA wheels may fail on some setups.'
        return torch.device('cuda'), gpu_note
    except Exception as e:
        return torch.device('cpu'), f'CUDA fallback: {type(e).__name__}: {e}'

set_seed(42)
device, device_note = pick_device()
print(build_runtime_note(device, device_note))
print(f'Runtime: {"Kaggle" if is_kaggle_runtime() else "Local"}')

# Set REQUIRE_CUDA=1 in Kaggle env vars to prevent accidental CPU training.
if os.environ.get('REQUIRE_CUDA', '0') == '1' and device.type != 'cuda':
    raise RuntimeError(
        'REQUIRE_CUDA=1 but CUDA is not usable. On Kaggle P100, run Cell 4 (P100 fix) with REINSTALL_TORCH_P100=True, restart session, then rerun.'
    )


## 1. BERT + BiLSTM Model

This section defines the hybrid architecture by combining BERT contextual embeddings, BiLSTM sequence modeling, attention, and a dense classifier head.

In [ ]:
class EnhancedBertForSequenceClassification(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name, low_cpu_mem_usage=False, device_map=None)
        self.dropout = nn.Dropout(dropout)
        h = self.bert.config.hidden_size         # 768
        self.lstm = nn.LSTM(h, 256, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.attention = nn.MultiheadAttention(512, num_heads=8, dropout=0.1, batch_first=True)
        self.layer_norm = nn.LayerNorm(512)
        self.classifier = nn.Sequential(
            nn.Linear(512,256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        seq = self.dropout(self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state)
        out = self.layer_norm(self.lstm(seq)[0])
        out, _ = self.attention(out, out, out)
        return self.classifier(torch.max(out, dim=1).values)

print("Model defined [OK]")


## 2. Dataset Class

This section builds a custom dataset class that tokenizes text inputs and returns tensors for input IDs, attention masks, and labels.

In [ ]:
class WELFakeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts, self.labels, self.tokenizer, self.max_length = texts, labels, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding='max_length',
                             max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(),
                'attention_mask': enc['attention_mask'].flatten(),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}
print("Dataset defined [OK]")


## 3. Load & Preprocess Data

This section loads the dataset, cleans and combines text fields, encodes labels, and creates stratified train, validation, and test splits.

In [ ]:
DATA_PATH = resolve_data_path()
print(f'Using dataset: {DATA_PATH}')
df = pd.read_csv(DATA_PATH)
if 'Unnamed: 0' in df.columns: df.drop('Unnamed: 0', axis=1, inplace=True)
df['title'] = df['title'].fillna('').astype(str)
df['text']  = df['text'].fillna('').astype(str)
df['enhanced_text'] = df['title'] + ' [SEP] ' + df['text'].str[:1000]
df['encoded_label'] = df['label'].astype(int)
df = df[df['enhanced_text'].str.len() > 10].reset_index(drop=True)

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['encoded_label'])
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['encoded_label'])
print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")


## 4. Config + DataLoaders

This section sets training hyperparameters, applies CPU-safe overrides when needed, computes class weights, and prepares balanced dataloaders.

In [ ]:
CONFIG = {
    'model_name': 'bert-base-uncased', 'max_length': 512,
    'batch_size': 8, 'learning_rate': 2e-5,
    'max_train_iterations': 1200, 'eval_interval': 100,
    'dropout': 0.3, 'warmup_ratio': 0.1, 'weight_decay': 0.01,
    'gradient_accumulation_steps': 4, 'patience': 3,
    'freeze_layers': 6, 'label_smoothing': 0.1,
}

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

# --- CPU mode overrides keep runtime manageable when CUDA is unavailable/incompatible ---
_on_cpu = (device.type == 'cpu')
if _on_cpu:
    CONFIG.update({
        'max_length': 256,
        'batch_size': 4,
        'max_train_iterations': 200,
        'eval_interval': 25,
        'gradient_accumulation_steps': 2,
    })
    print('CPU mode overrides applied:', {
        'max_length': CONFIG['max_length'],
        'batch_size': CONFIG['batch_size'],
        'max_train_iterations': CONFIG['max_train_iterations'],
        'eval_interval': CONFIG['eval_interval'],
        'gradient_accumulation_steps': CONFIG['gradient_accumulation_steps'],
    })

# --- CPU subset: smaller data when no GPU so runtime stays manageable ---
if _on_cpu:
    train_df = train_df.sample(n=min(5000, len(train_df)), random_state=42).reset_index(drop=True)
    val_df   = val_df.sample(n=min(1000,  len(val_df)),   random_state=42).reset_index(drop=True)
    print(f'CPU subset -> Train: {len(train_df)}  Val: {len(val_df)}')

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced',
        classes=np.unique(train_df['encoded_label']),
        y=train_df['encoded_label'])
class_weights = torch.tensor(cw, dtype=torch.float).to(device)
print(f'Class weights: Real={cw[0]:.3f}  Fake={cw[1]:.3f}')

train_ds = WELFakeDataset(train_df['enhanced_text'].values, train_df['encoded_label'].values, tokenizer, CONFIG['max_length'])
val_ds   = WELFakeDataset(val_df['enhanced_text'].values,   val_df['encoded_label'].values,   tokenizer, CONFIG['max_length'])

from torch.utils.data import WeightedRandomSampler
sampler = WeightedRandomSampler([cw[l] for l in train_df['encoded_label']], len(train_df), replacement=True)

# Kaggle + notebook debugger can trigger multiprocessing worker cleanup assertions.
# Use single-process data loading in Kaggle/CPU for stable long runs.
_is_kaggle = is_kaggle_runtime()
_nw = 0 if (_on_cpu or _is_kaggle) else 2
_pm = not _on_cpu
if _is_kaggle and _nw == 0:
    print('Kaggle runtime detected: forcing num_workers=0 to avoid DataLoader worker shutdown assertions.')

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], sampler=sampler, num_workers=_nw, pin_memory=_pm)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,  num_workers=_nw, pin_memory=_pm)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Effective batch size: {CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"]}')


## 5. Training & Eval Functions

This section defines reusable functions that train the model, evaluate metrics, and generate probability outputs for deeper diagnostics.

In [ ]:
def train_for_iterations(model, loader, optimizer, scheduler, criterion, device, accum, max_iterations, eval_interval, val_loader, start_iter=0):
    model.train()
    optimizer.zero_grad()
    iter_steps, iter_losses, iter_accs = [], [], []
    checkpoint_steps = []
    checkpoint_train_loss, checkpoint_train_acc = [], []
    val_loss_history, val_acc_history, val_f1_history = [], [], []
    val_precision_history, val_recall_history = [], []
    running_correct = running_total = 0
    checkpoint_loss_sum = 0.0
    checkpoint_batches = 0
    best_f1, best_state, patience_ctr = 0.0, None, 0
    data_iter = iter(loader)

    for global_iter in range(start_iter + 1, max_iterations + 1):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            batch = next(data_iter)

        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)

        logits = model(input_ids=ids, attention_mask=mask)
        batch_loss = criterion(logits, lbls)
        (batch_loss / accum).backward()

        if global_iter % accum == 0 or global_iter == max_iterations:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        batch_correct = (logits.detach().argmax(1) == lbls).sum().item()
        running_correct += batch_correct
        running_total += lbls.size(0)
        checkpoint_loss_sum += batch_loss.item()
        checkpoint_batches += 1

        iter_steps.append(global_iter)
        iter_losses.append(batch_loss.item())
        iter_accs.append(running_correct / running_total)

        if global_iter % eval_interval == 0 or global_iter == max_iterations:
            val = evaluate(model, val_loader, criterion, device)
            checkpoint_steps.append(global_iter)
            checkpoint_train_loss.append(checkpoint_loss_sum / max(1, checkpoint_batches))
            checkpoint_train_acc.append(running_correct / running_total)
            val_loss_history.append(val['loss'])
            val_acc_history.append(val['accuracy'])
            val_f1_history.append(val['f1'])
            val_precision_history.append(val['precision'])
            val_recall_history.append(val['recall'])

            print(f"  Step {global_iter}: train loss={checkpoint_train_loss[-1]:.4f} acc={checkpoint_train_acc[-1]:.4f}")
            print(f"             val   loss={val['loss']:.4f} acc={val['accuracy']:.4f} F1={val['f1']:.4f}")

            if val['f1'] > best_f1:
                best_f1 = val['f1']
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}
                patience_ctr = 0
                print(f"  [Best F1]: {best_f1:.4f}")
            else:
                patience_ctr += 1
                if patience_ctr >= CONFIG['patience']:
                    print('Early stopping')
                    break

            checkpoint_loss_sum = 0.0
            checkpoint_batches = 0
            model.train()

    if best_state is None:
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    history = {
        'iter_train_step': iter_steps,
        'iter_train_loss': iter_losses,
        'iter_train_acc': iter_accs,
        'checkpoint_step': checkpoint_steps,
        'checkpoint_train_loss': checkpoint_train_loss,
        'checkpoint_train_acc': checkpoint_train_acc,
        'val_loss': val_loss_history,
        'val_acc': val_acc_history,
        'val_f1': val_f1_history,
        'val_precision': val_precision_history,
        'val_recall': val_recall_history,
    }
    return history, best_state, best_f1, global_iter

def evaluate(model, loader, criterion, device):
    model.eval(); total_loss=0; preds, labels=[], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            ids=batch['input_ids'].to(device); mask=batch['attention_mask'].to(device)
            lbls=batch['labels'].to(device)
            out=model(input_ids=ids, attention_mask=mask)
            total_loss+=criterion(out,lbls).item()
            preds.extend(out.argmax(1).cpu().numpy()); labels.extend(lbls.cpu().numpy())
    f1=f1_score(labels,preds,average='weighted')
    p,r,*_=precision_recall_fscore_support(labels,preds,average='weighted')
    return dict(loss=total_loss/len(loader), accuracy=accuracy_score(labels,preds), f1=f1, precision=p, recall=r)

def predict_with_scores(model, loader, device):
    model.eval(); preds, labels, probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Predicting'):
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            logits = model(input_ids=ids, attention_mask=mask)
            soft = torch.softmax(logits, dim=1)
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            probs.extend(soft[:, 1].cpu().numpy())
            labels.extend(lbls.cpu().numpy())
    return np.array(labels), np.array(preds), np.array(probs)

print("Functions defined [OK]")


## 6. Iteration-Based Training Loop

This section trains the hybrid model for a fixed number of optimizer iterations, evaluates it at regular iteration checkpoints, applies early stopping, and restores the best-performing checkpoint.

In [ ]:
model = EnhancedBertForSequenceClassification(
    CONFIG['model_name'], num_classes=2, dropout=CONFIG['dropout']
 ).to(device)

for p in model.bert.embeddings.parameters(): p.requires_grad = False
for i, layer in enumerate(model.bert.encoder.layer):
    if i < CONFIG['freeze_layers']:
        for p in layer.parameters(): p.requires_grad = False

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=CONFIG['label_smoothing'])
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
total_steps = max(1, (CONFIG['max_train_iterations'] + CONFIG['gradient_accumulation_steps'] - 1) // CONFIG['gradient_accumulation_steps'])
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * CONFIG['warmup_ratio']), total_steps)

history, best_state, best_f1, completed_iterations = train_for_iterations(
    model, train_loader, optimizer, scheduler, criterion, device,
    CONFIG['gradient_accumulation_steps'], CONFIG['max_train_iterations'], CONFIG['eval_interval'], val_loader
 )

model.load_state_dict({k:v.to(device) for k,v in best_state.items()})
print(f"\nDone | Best Val F1: {best_f1:.4f} | Completed iterations: {completed_iterations}")


## 7. Training Plots

This section visualizes training dynamics with iteration-based curves for loss, accuracy, validation metrics, and generalization gap at evaluation checkpoints.

In [ ]:
# Distinct color palette for training-history metrics
C = {
    'train_loss': '#e76f51',
    'val_loss': '#264653',
    'train_acc': '#2a9d8f',
    'val_acc': '#f4a261',
    'val_f1': '#457b9d',
    'val_precision': '#8d99ae',
    'val_recall': '#e63946'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
it = np.array(history['iter_train_step'])
ct = np.array(history['checkpoint_step'])

# 1) Loss curves by iteration
axes[0, 0].plot(it, history['iter_train_loss'], color=C['train_loss'], lw=1.6, alpha=0.65, label='Train Loss (per iteration)')
axes[0, 0].plot(ct, history['checkpoint_train_loss'], color=C['val_loss'], marker='o', lw=2.2, label='Train Loss (checkpoint average)')
axes[0, 0].set_title('Hybrid Model: Loss vs Iteration')
axes[0, 0].set_xlabel('Iteration'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(alpha=.25); axes[0, 0].legend()

# 2) Accuracy curves by iteration
axes[0, 1].plot(it, history['iter_train_acc'], color=C['train_acc'], lw=1.6, alpha=0.65, label='Train Accuracy (running)')
axes[0, 1].plot(ct, history['checkpoint_train_acc'], color=C['val_acc'], marker='o', lw=2.2, label='Train Accuracy (checkpoint)')
axes[0, 1].set_title('Hybrid Model: Accuracy vs Iteration')
axes[0, 1].set_xlabel('Iteration'); axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_ylim(0, 1.02)
axes[0, 1].grid(alpha=.25); axes[0, 1].legend()

# 3) Validation class-balance metrics (mapped to iteration checkpoints)
axes[1, 0].plot(ct, history['val_f1'], color=C['val_f1'], marker='o', lw=2, label='Val F1')
axes[1, 0].plot(ct, history['val_precision'], color=C['val_precision'], marker='s', lw=2, label='Val Precision')
axes[1, 0].plot(ct, history['val_recall'], color=C['val_recall'], marker='^', lw=2, label='Val Recall')
axes[1, 0].set_title('Hybrid Model: Validation Metrics vs Iteration')
axes[1, 0].set_xlabel('Iteration'); axes[1, 0].set_ylabel('Score')
axes[1, 0].set_ylim(0, 1.02)
axes[1, 0].grid(alpha=.25); axes[1, 0].legend()

# 4) Generalization gap (accuracy) at validation checkpoints
acc_gap = np.array(history['checkpoint_train_acc']) - np.array(history['val_acc'])
axes[1, 1].bar(ct, acc_gap, color='#6d597a', width=max(1, len(ct) // 20), alpha=0.85)
axes[1, 1].axhline(0, color='black', lw=1)
axes[1, 1].set_title('Hybrid Model: Train-Val Accuracy Gap vs Iteration')
axes[1, 1].set_xlabel('Iteration'); axes[1, 1].set_ylabel('Gap (Train - Val)')
axes[1, 1].grid(axis='y', alpha=.25)

plt.suptitle('Training Performance Dashboard (Hybrid BERT+BiLSTM)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('hybrid_training_dashboard_iterations.png', dpi=140, bbox_inches='tight')
plt.show()


## 7.5 Hybrid Model Diagnostics

This section adds deeper diagnostics for the hybrid model, including confusion matrix, ROC curve, precision-recall curve, and probability distribution by class.

In [ ]:
# --- Hybrid diagnostics on validation data ---
y_true, y_pred, y_prob = predict_with_scores(model, val_loader, device)

cm = confusion_matrix(y_true, y_pred)
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)
prec, rec, _ = precision_recall_curve(y_true, y_prob)

diag_colors = {
    'real': '#1f77b4',
    'fake': '#d62728',
    'roc': '#ff7f0e',
    'pr': '#2ca02c',
    'chance': '#6c757d'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1) Confusion matrix
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0, 0],
    xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake']
    )
axes[0, 0].set_title('Hybrid Model: Confusion Matrix (Validation)', fontweight='bold')
axes[0, 0].set_xlabel('Predicted Label')
axes[0, 0].set_ylabel('True Label')

# 2) ROC curve
axes[0, 1].plot(fpr, tpr, color=diag_colors['roc'], lw=2.5, label=f'ROC AUC = {roc_auc:.4f}')
axes[0, 1].plot([0, 1], [0, 1], linestyle='--', color=diag_colors['chance'], label='Chance')
axes[0, 1].set_title('Hybrid Model: ROC Curve (Validation)', fontweight='bold')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].grid(alpha=0.25)
axes[0, 1].legend()

# 3) Precision-recall curve
axes[1, 0].plot(rec, prec, color=diag_colors['pr'], lw=2.5)
axes[1, 0].set_title('Hybrid Model: Precision-Recall Curve (Validation)', fontweight='bold')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].grid(alpha=0.25)

# 4) Predicted probability distribution by class
axes[1, 1].hist(
    y_prob[y_true == 0], bins=30, alpha=0.65, color=diag_colors['real'],
    label='True Real (label=0)', density=True
    )
axes[1, 1].hist(
    y_prob[y_true == 1], bins=30, alpha=0.65, color=diag_colors['fake'],
    label='True Fake (label=1)', density=True
    )
axes[1, 1].axvline(0.5, color=diag_colors['chance'], linestyle='--', lw=1.5, label='Threshold 0.5')
axes[1, 1].set_title('Hybrid Model: Prediction Score Distribution', fontweight='bold')
axes[1, 1].set_xlabel('Predicted Probability of Fake')
axes[1, 1].set_ylabel('Density')
axes[1, 1].grid(alpha=0.25)
axes[1, 1].legend()

plt.suptitle('Hybrid Model Diagnostic Dashboard', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('hybrid_diagnostic_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()

print('\nClassification Report (Validation):')
print(classification_report(y_true, y_pred, digits=4))


## 8. Traditional ML Baselines

This section trains classical baseline models on TF-IDF features and records their performance for a fair comparison.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

# Use full training data for ML models (they train fast)
vec   = TfidfVectorizer(max_features=30000, sublinear_tf=True, ngram_range=(1,2))
X_tr  = vec.fit_transform(train_df['enhanced_text'])
X_val = vec.transform(val_df['enhanced_text'])
y_tr  = train_df['encoded_label'].values
y_val = val_df['encoded_label'].values

ml_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Linear SVC':          LinearSVC(max_iter=2000, random_state=42),
    'Naive Bayes':         MultinomialNB(),
}

results = []
for name, clf in ml_models.items():
    t0=time.time(); clf.fit(X_tr, y_tr); preds=clf.predict(X_val)
    p,r,*_=precision_recall_fscore_support(y_val,preds,average='weighted')
    results.append(dict(Model=name, Accuracy=accuracy_score(y_val,preds),
                        F1=f1_score(y_val,preds,average='weighted'),
                        Precision=p, Recall=r))
    print(f"{name:22s}  Acc={results[-1]['Accuracy']:.4f}  F1={results[-1]['F1']:.4f}  ({time.time()-t0:.1f}s)")

results.append(dict(Model='BERT+BiLSTM (Hybrid)',
    Accuracy=history['val_acc'][-1], F1=history['val_f1'][-1],
    Precision=history['val_precision'][-1], Recall=history['val_recall'][-1]))

results_df = pd.DataFrame(results)
print("\n", results_df[['Model','Accuracy','F1','Precision','Recall']].to_string(index=False))


## 9. Model Comparison

This section compares all models with consistent color mapping across charts so you can evaluate relative strengths quickly.

In [ ]:
# Model comparison dashboard with fixed, highly distinct colors per model
model_order = results_df['Model'].tolist()

# Colorblind-friendly, high-contrast palette (cycled if models > palette length)
distinct_palette = [
    '#1f77b4',  # blue
    '#ff7f0e',  # orange
    '#2ca02c',  # green
    '#d62728',  # red
    '#9467bd',  # purple
    '#8c564b',  # brown
    '#e377c2',  # pink
    '#7f7f7f',  # gray
    '#bcbd22',  # olive
    '#17becf'   # cyan
]
model_colors = {m: distinct_palette[i % len(distinct_palette)] for i, m in enumerate(model_order)}

# Keep explicit model color assignments for consistency across reruns
model_colors.update({
    'Logistic Regression': '#1f77b4',
    'Random Forest': '#2ca02c',
    'Linear SVC': '#ff7f0e',
    'Naive Bayes': '#9467bd',
    'BERT+BiLSTM (Hybrid)': '#d62728'
})

print('Model color mapping:')
for m in model_order:
    print(f'  {m}: {model_colors[m]}')

fig, axes = plt.subplots(2, 2, figsize=(18, 11))

# 1) Grouped metric bars (each model has a unique color)
melted = results_df.melt(
    id_vars='Model',
    value_vars=['Accuracy', 'F1', 'Precision', 'Recall'],
    var_name='Metric',
    value_name='Score'
    )
sns.barplot(
    data=melted, x='Metric', y='Score', hue='Model',
    hue_order=model_order, palette=model_colors, ax=axes[0, 0]
    )
axes[0, 0].set_title('Metric Comparison Across Models', fontweight='bold')
axes[0, 0].set_ylim(0.4, 1.02)
axes[0, 0].grid(axis='y', alpha=0.25)
axes[0, 0].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')

# 2) Accuracy ranking with the same model colors
rank_df = results_df.sort_values('Accuracy', ascending=True)
bar_colors = [model_colors[m] for m in rank_df['Model']]
bars = axes[0, 1].barh(rank_df['Model'], rank_df['Accuracy'], color=bar_colors, alpha=0.9)
for bar, val in zip(bars, rank_df['Accuracy']):
    axes[0, 1].text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)
axes[0, 1].set_title('Accuracy Ranking', fontweight='bold')
axes[0, 1].set_xlim(0.4, 1.02)
axes[0, 1].grid(axis='x', alpha=0.25)

# 3) Metric heatmap (numeric readability)
metric_table = results_df.set_index('Model')[['Accuracy', 'F1', 'Precision', 'Recall']]
sns.heatmap(metric_table, annot=True, fmt='.4f', cmap='YlGnBu', cbar=True, ax=axes[1, 0])
axes[1, 0].set_title('Performance Heatmap', fontweight='bold')
axes[1, 0].set_xlabel('Metric'); axes[1, 0].set_ylabel('Model')

# 4) Precision vs Recall scatter (same model colors)
for _, row in results_df.iterrows():
    m = row['Model']
    p = row['Precision']
    r = row['Recall']
    axes[1, 1].scatter(p, r, s=170, color=model_colors[m], edgecolor='black', alpha=0.9, label=m)
    axes[1, 1].text(p + 0.0008, r + 0.0008, m, fontsize=8)
axes[1, 1].set_title('Precision vs Recall', fontweight='bold')
axes[1, 1].set_xlabel('Precision'); axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_xlim(0.4, 1.02); axes[1, 1].set_ylim(0.4, 1.02)
axes[1, 1].grid(alpha=0.25)

# De-duplicate legend entries on scatter plot
handles, labels = axes[1, 1].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axes[1, 1].legend(by_label.values(), by_label.keys(), title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.suptitle('Traditional ML vs Hybrid BERT+BiLSTM (Distinct Model Colors)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison_distinct_colors.png', dpi=140, bbox_inches='tight')
plt.show()


## 10. Unified Performance Dashboard

This section builds one consolidated dashboard that combines hybrid-model diagnostics and hybrid-vs-traditional comparison views in a single run.

In [ ]:
# --- Unified dashboard: hybrid diagnostics + model comparison ---
y_true_u, y_pred_u, y_prob_u = predict_with_scores(model, val_loader, device)
cm_u = confusion_matrix(y_true_u, y_pred_u)
fpr_u, tpr_u, _ = roc_curve(y_true_u, y_prob_u)
roc_auc_u = auc(fpr_u, tpr_u)
prec_u, rec_u, _ = precision_recall_curve(y_true_u, y_prob_u)

# Keep consistent model colors even if this cell is run independently
model_order_u = results_df['Model'].tolist()
distinct_palette_u = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]
model_colors_u = {m: distinct_palette_u[i % len(distinct_palette_u)] for i, m in enumerate(model_order_u)}
model_colors_u.update({
    'Logistic Regression': '#1f77b4',
    'Random Forest': '#2ca02c',
    'Linear SVC': '#ff7f0e',
    'Naive Bayes': '#9467bd',
    'BERT+BiLSTM (Hybrid)': '#d62728'
})

fig, axes = plt.subplots(3, 2, figsize=(19, 16))

# 1) Hybrid confusion matrix
sns.heatmap(
    cm_u, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0, 0],
    xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake']
    )
axes[0, 0].set_title('Hybrid: Confusion Matrix (Validation)', fontweight='bold')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('True')

# 2) Hybrid ROC
axes[0, 1].plot(fpr_u, tpr_u, color='#ff7f0e', lw=2.5, label=f'ROC AUC = {roc_auc_u:.4f}')
axes[0, 1].plot([0, 1], [0, 1], linestyle='--', color='#6c757d', label='Chance')
axes[0, 1].set_title('Hybrid: ROC Curve', fontweight='bold')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].grid(alpha=0.25)
axes[0, 1].legend()

# 3) Hybrid precision-recall
axes[1, 0].plot(rec_u, prec_u, color='#2ca02c', lw=2.5)
axes[1, 0].set_title('Hybrid: Precision-Recall Curve', fontweight='bold')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].grid(alpha=0.25)

# 4) Hybrid score distribution
axes[1, 1].hist(
    y_prob_u[y_true_u == 0], bins=30, alpha=0.65, color='#1f77b4',
    label='True Real (0)', density=True
    )
axes[1, 1].hist(
    y_prob_u[y_true_u == 1], bins=30, alpha=0.65, color='#d62728',
    label='True Fake (1)', density=True
    )
axes[1, 1].axvline(0.5, color='#6c757d', linestyle='--', lw=1.5, label='Threshold 0.5')
axes[1, 1].set_title('Hybrid: Prediction Score Distribution', fontweight='bold')
axes[1, 1].set_xlabel('Predicted Probability of Fake')
axes[1, 1].set_ylabel('Density')
axes[1, 1].grid(alpha=0.25)
axes[1, 1].legend()

# 5) Model metric comparison
melted_u = results_df.melt(
    id_vars='Model',
    value_vars=['Accuracy', 'F1', 'Precision', 'Recall'],
    var_name='Metric',
    value_name='Score'
    )
sns.barplot(
    data=melted_u, x='Metric', y='Score', hue='Model',
    hue_order=model_order_u, palette=model_colors_u, ax=axes[2, 0]
    )
axes[2, 0].set_title('Models: Metric Comparison', fontweight='bold')
axes[2, 0].set_ylim(0.4, 1.02)
axes[2, 0].grid(axis='y', alpha=0.25)
axes[2, 0].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')

# 6) Accuracy ranking
rank_u = results_df.sort_values('Accuracy', ascending=True)
bars_u = axes[2, 1].barh(
    rank_u['Model'], rank_u['Accuracy'],
    color=[model_colors_u[m] for m in rank_u['Model']], alpha=0.9
    )
for b, v in zip(bars_u, rank_u['Accuracy']):
    axes[2, 1].text(v + 0.003, b.get_y() + b.get_height() / 2, f'{v:.4f}', va='center', fontsize=9)
axes[2, 1].set_title('Models: Accuracy Ranking', fontweight='bold')
axes[2, 1].set_xlim(0.4, 1.02)
axes[2, 1].grid(axis='x', alpha=0.25)

plt.suptitle('TruthLens Unified Performance Dashboard', fontsize=17, fontweight='bold')
plt.tight_layout()
plt.savefig('truthlens_unified_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Save Model

This section exports model weights, tokenizer files, metric tables, and training history so you can reuse results in deployment.

In [ ]:
import json as _json
SAVE = resolve_save_dir()
os.makedirs(SAVE, exist_ok=True)
torch.save(model.state_dict(), f'{SAVE}/model_weights.pth')
tokenizer.save_pretrained(SAVE)
results_df.to_csv(f'{SAVE}/comparison.csv', index=False)
with open(f'{SAVE}/history.json','w') as f: _json.dump(history, f)
print(f"Saved to {SAVE}:")
for fn in sorted(os.listdir(SAVE)):
    print(f"  {fn}  ({os.path.getsize(os.path.join(SAVE,fn))/1e6:.2f} MB)")
